#  DATA PREPARATION

I dette prosjektet vil jeg se om det er mulig å forutsi hvem som vinner et sjakkparti, bare basert på informasjon vi har **før** partiet starter.

Dataene kommer fra Lichess sin åpne database, som inneholder alle partier som er spilt på lichess.org. Databasen ligger også på Hugging Face, og er lisensiert som CC0, så den er fri å bruke.

- https://database.lichess.org/
- https://huggingface.co/datasets/Lichess/standard-chess-games

Én måned med partier er allerede over 200 000 partier, så jeg bruker bare juni 2013 og tar et tilfeldig utvalg på 20 000 partier. Det er mer enn nok for modellene jeg skal teste, og det gjør at notebooken kjører raskt.

In [1]:
import os
import urllib.request
import numpy as np
import pandas as pd

In [2]:
# Laster ned rådataene én gang og lagrer dem lokalt i dataset/raw/.
# Filen er ca. 70 MB, så den ligger ikke på GitHub (se .gitignore).
# Hvis filen allerede finnes, hopper vi over nedlastingen.

raw_url = "https://huggingface.co/datasets/Lichess/standard-chess-games/resolve/main/data/year=2013/month=06/train-00000-of-00001.parquet"
raw_path = "../dataset/raw/lichess_2013_06.parquet"

os.makedirs("../dataset/raw", exist_ok=True)
if not os.path.exists(raw_path):
    print("Laster ned ...")
    urllib.request.urlretrieve(raw_url, raw_path)

print(f"Filstørrelse: {os.path.getsize(raw_path) / 1_000_000:.1f} MB")

Filstørrelse: 69.2 MB


In [3]:
# Parquet er et kolonnebasert filformat, mye mindre enn CSV for så store datasett.
# pandas kan lese det direkte (krever pyarrow).
raw = pd.read_parquet(raw_path)
print(f"{raw.shape[0]} rows, {raw.shape[1]} columns")

224679 rows, 18 columns


In [4]:
# Ser på de første radene. movetext er hele partiet skrevet som trekk, så den er veldig lang.
raw.drop(columns=["movetext"]).head(5)

,Event,Site,White,Black,Result,WhiteTitle,BlackTitle,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,UTCDate,UTCTime,ECO,Opening,Termination,TimeControl
0,Rated Bullet game,https://lichess.org/in28emmw,Kazuma,kikeillana,1-0,NaN,NaN,1756.0,1684.0,11.0,-9.0,2013-05-31,22:00:22,A07,King's Indian Attack: Keres Variation #2,Normal,60+0
1,Rated Bullet game,https://lichess.org/e174t8h7,Aceves,calculus,0-1,NaN,NaN,1487.0,1568.0,-9.0,9.0,2013-05-31,22:02:13,D00,Queen's Pawn Game #3,Time forfeit,60+1
2,Rated Blitz game,https://lichess.org/d4ui60z6,melro,patrimpas,0-1,NaN,NaN,1144.0,1912.0,-1.0,0.0,2013-05-31,22:02:15,B20,Sicilian Defense: Staunton-Cochrane Variation,Normal,240+0
3,Rated Blitz game,https://lichess.org/gx3qb4ur,Igortroufignofski,Ndrina,0-1,NaN,NaN,1155.0,1096.0,-13.0,16.0,2013-05-31,22:02:28,A07,King's Indian Attack,Normal,300+3
4,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/kxhslrt4,panickat,prob4,0-1,NaN,NaN,1258.0,1459.0,-11.0,6.0,2013-05-31,22:02:14,B20,Sicilian Defense: Bowdler Attack,Normal,300+2


In [5]:
raw["movetext"].iloc[0][:300]

'1. Nf3 d5 2. g3 Bg4 3. Bg2 Bxf3 4. Bxf3 e6 5. O-O Bb4 6. d4 Nd7 7. c3 Ba5 8. Bf4 Bb6 9. b4 a6 10. a4 c6 11. Nd2 Ngf6 12. e4 dxe4 13. Nxe4 Nxe4 14. Bxe4 f6 15. c4 h6 16. c5 Bc7 17. Qb3 Bxf4 18. Qxe6+ Qe7 19. Bg6+ Kd8 20. Qxe7+ Kxe7 21. gxf4 Rhe8 22. Bxe8 Rxe8 23. Rfe1+ Kf7 24. Rxe8 Kxe8 25. Re1+ Kf7 '

#### Results :
Hver rad er ett parti. Vi har informasjon om spillerne (brukernavn og rating), resultatet, åpningen, tidskontrollen og hvordan partiet endte. I tillegg har vi `movetext`, som er alle trekkene i partiet.

Noen av kolonnene er ikke relevante for prosjektet, eller bør ikke være med:

- `White` og `Black` er brukernavn. Det er personopplysninger, og modellen skal uansett ikke lære hvem som spiller, så jeg fjerner dem.
- `Site` er en lenke til partiet. Den er unik for hvert parti og gir ingen informasjon.
- `WhiteTitle` og `BlackTitle` er nesten alltid tomme (titler som GM og IM), så de fjernes også.
- `UTCDate` er nesten den samme for alle partiene siden alt er fra samme måned.

In [6]:
print(raw[["WhiteTitle", "BlackTitle"]].notna().mean().round(4))
print()
print(raw["Event"].value_counts().head(8))

WhiteTitle    0.0002
BlackTitle    0.0002
dtype: float64

Event
Rated Blitz game                                                   86804
Rated Classical game                                               76031
Rated Bullet game                                                  56103
Rated Correspondence game                                            567
Rated Bullet tournament https://lichess.org/tournament/9dxuago2       41
Rated Bullet tournament https://lichess.org/tournament/q6gxga0x       40
Rated Bullet tournament https://lichess.org/tournament/vivrwvf2       39
Rated Bullet tournament https://lichess.org/tournament/vzavejgx       37
Name: count, dtype: int64


Under 1 % av partiene har en tittel registrert, så de kolonnene er ikke nyttige.

`Event` inneholder typen parti (Bullet, Blitz, Classical, Correspondence), men turneringspartier har også en lenke til turneringen i teksten. Jeg henter derfor bare ut selve spilltypen.

In [7]:
games = raw.copy()

# Henter ut spilltypen fra Event-teksten, f.eks. "Rated Blitz game" -> "Blitz"
games["game_type"] = games["Event"].str.extract(r"(Bullet|Blitz|Rapid|Classical|Correspondence)", expand=False)

# Tidskontroll er skrevet som "300+8": 300 sekunder på klokka og 8 sekunder ekstra per trekk.
# Correspondence-partier har "-" fordi de ikke har klokke.
time_split = games["TimeControl"].str.split("+", expand=True)
games["base_time"] = pd.to_numeric(time_split[0], errors="coerce")
games["increment"] = pd.to_numeric(time_split[1], errors="coerce")

# Antall trekk: finner det høyeste trekknummeret i movetext, f.eks. "34." -> 34
games["num_moves"] = games["movetext"].str.findall(r"(\d+)\.").apply(lambda m: max(map(int, m)) if len(m) > 0 else 0)

# Åpningsfamilien er det som står før kolon: "Sicilian Defense: Najdorf Variation" -> "Sicilian Defense"
# Noen åpninger har også et nummer på slutten ("Queen's Pawn Game #3"), det fjerner jeg også.
games["opening_family"] = games["Opening"].str.split(":").str[0].str.replace(r"\s*#\d+$", "", regex=True).str.strip()

# Timen partiet startet (UTC), i tilfelle tidspunktet på døgnet har noe å si
games["hour"] = pd.to_datetime(games["UTCTime"].astype(str), format="%H:%M:%S").dt.hour

games[["Event", "game_type", "TimeControl", "base_time", "increment", "num_moves", "Opening", "opening_family", "hour"]].head(10)

,Event,game_type,TimeControl,base_time,increment,num_moves,Opening,opening_family,hour
0,Rated Bullet game,Bullet,60+0,60.0,0.0,32,King's Indian Attack: Keres Variation #2,King's Indian Attack,22
1,Rated Bullet game,Bullet,60+1,60.0,1.0,34,Queen's Pawn Game #3,Queen's Pawn Game,22
2,Rated Blitz game,Blitz,240+0,240.0,0.0,30,Sicilian Defense: Staunton-Cochrane Variation,Sicilian Defense,22
3,Rated Blitz game,Blitz,300+3,300.0,3.0,25,King's Indian Attack,King's Indian Attack,22
4,Rated Blitz tournament https://lichess.org/tou...,Blitz,300+2,300.0,2.0,19,Sicilian Defense: Bowdler Attack,Sicilian Defense,22
5,Rated Bullet game,Bullet,120+0,120.0,0.0,38,Owen Defense,Owen Defense,22
6,Rated Blitz game,Blitz,300+2,300.0,2.0,31,"Italian Game: Two Knights Defense, Modern Bish...",Italian Game,22
7,Rated Blitz game,Blitz,180+1,180.0,1.0,19,Queen's Pawn,Queen's Pawn,22
8,Rated Classical game,Classical,660+1,660.0,1.0,19,Alekhine Defense: Normal Variation,Alekhine Defense,22
9,Rated Bullet game,Bullet,120+0,120.0,0.0,19,Bishop's Opening: Boi Variation,Bishop's Opening,22


In [8]:
# Jeg gir kolonnene nye navn i snake_case, samme stil som i forrige prosjekt.
games = games.rename(columns={
    "WhiteElo": "white_elo",
    "BlackElo": "black_elo",
    "WhiteRatingDiff": "white_rating_change",
    "BlackRatingDiff": "black_rating_change",
    "ECO": "eco",
    "Opening": "opening",
    "Termination": "termination",
    "TimeControl": "time_control",
    "Result": "result",
})

columns_to_keep = ["game_type", "time_control", "base_time", "increment", "hour",
                   "white_elo", "black_elo", "eco", "opening", "opening_family",
                   "num_moves", "termination", "white_rating_change", "black_rating_change", "result"]
games = games[columns_to_keep]
games.head()

,game_type,time_control,base_time,increment,hour,white_elo,black_elo,eco,opening,opening_family,num_moves,termination,white_rating_change,black_rating_change,result
0,Bullet,60+0,60.0,0.0,22,1756.0,1684.0,A07,King's Indian Attack: Keres Variation #2,King's Indian Attack,32,Normal,11.0,-9.0,1-0
1,Bullet,60+1,60.0,1.0,22,1487.0,1568.0,D00,Queen's Pawn Game #3,Queen's Pawn Game,34,Time forfeit,-9.0,9.0,0-1
2,Blitz,240+0,240.0,0.0,22,1144.0,1912.0,B20,Sicilian Defense: Staunton-Cochrane Variation,Sicilian Defense,30,Normal,-1.0,0.0,0-1
3,Blitz,300+3,300.0,3.0,22,1155.0,1096.0,A07,King's Indian Attack,King's Indian Attack,25,Normal,-13.0,16.0,0-1
4,Blitz,300+2,300.0,2.0,22,1258.0,1459.0,B20,Sicilian Defense: Bowdler Attack,Sicilian Defense,19,Normal,-11.0,6.0,0-1


In [9]:
# Tar et tilfeldig utvalg på 20 000 partier.
# random_state=13 gjør at vi får det samme utvalget hver gang notebooken kjøres.
random_state = 13
chess_sample = games.sample(n=20_000, random_state=random_state).reset_index(drop=True)

# Sjekker at fordelingen av resultatene er omtrent den samme i utvalget som i hele måneden
print("Hele måneden:")
print(games["result"].value_counts(normalize=True).round(3))
print("\nUtvalget:")
print(chess_sample["result"].value_counts(normalize=True).round(3))

Hele måneden:
result
1-0        0.510
0-1        0.457
1/2-1/2    0.032
Name: proportion, dtype: float64

Utvalget:
result
1-0        0.516
0-1        0.450
1/2-1/2    0.034
Name: proportion, dtype: float64


In [10]:
chess_sample.to_csv("../dataset/chess_games.csv", index=False)
print(f"Lagret {chess_sample.shape[0]} rader og {chess_sample.shape[1]} kolonner til dataset/chess_games.csv")

Lagret 20000 rader og 15 kolonner til dataset/chess_games.csv


#### Results :
Utvalget har nesten nøyaktig samme fordeling av hvit seier, svart seier og remis som hele måneden. Det betyr at utvalget er representativt, og at vi ikke har fått en skjev del av dataene bare ved et uhell.

Datasettet er nå lagret som `dataset/chess_games.csv`, og resten av prosjektet bruker bare denne filen (se `chess.ipynb`).

Merk: `num_moves`, `termination`, `white_rating_change` og `black_rating_change` er informasjon vi først vet **etter** at partiet er ferdig. Jeg beholder dem i CSV-filen fordi de er interessante i EDA-en, men de kan ikke brukes som features i modellen. Mer om det i preprocessing.